## Iteration 8 — MMR + Cross-Encoder Reranking

Pipeline: FAISS Top-10 → MMR Top-5 → Cross-Encoder → final Top-5

Iteration 7 achieved 100% Any Top-5. This experiment tests whether reranking the same MMR-selected five chunks can improve Top-1/Top-3 without losing Top-5 coverage.

Document representation remains exactly the Iteration 2 representation: category + section + content.


In [1]:
!pip -q install sentence-transformers faiss-cpu pandas numpy


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from IPython.display import display

chunks_df = pd.read_csv("./02_rmit_assessment_support_chunks.csv")
with open("./test_prompts.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)
test_df = pd.DataFrame(test_data["tests"])

print("Chunks:", len(chunks_df))
print("Test prompts:", len(test_df))

c:\Users\KRUSHIL\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunks: 71
Test prompts: 12


In [3]:
chunks_df["embedding_text"] = (
    chunks_df["category"].fillna("") + ". " +
    chunks_df["section"].fillna("") + ". " +
    chunks_df["content"].fillna("")
)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(
    chunks_df["embedding_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(embeddings)
faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("FAISS vectors:", faiss_index.ntotal)
print("Embedding dimension:", embeddings.shape[1])

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 19715.37it/s]


FAISS vectors: 71
Embedding dimension: 384


In [4]:
def faiss_candidates(question, k=10):
    q = embedding_model.encode(
        [question], convert_to_numpy=True
    ).astype("float32")
    faiss.normalize_L2(q)
    scores, indices = faiss_index.search(q, k)
    return q[0], [
        {"index": int(idx), "faiss_rank": rank + 1,
         "faiss_score": float(score)}
        for rank, (idx, score) in enumerate(zip(indices[0], scores[0]))
    ]


def mmr_select(question, candidate_k=10, final_k=5, lambda_param=0.70):
    q, candidates = faiss_candidates(question, candidate_k)
    candidate_indices = [x["index"] for x in candidates]
    candidate_embeddings = embeddings[candidate_indices]
    relevance = np.dot(candidate_embeddings, q.reshape(-1, 1)).flatten()

    selected, remaining = [], list(range(len(candidate_indices)))

    while remaining and len(selected) < final_k:
        best_pos, best_score = None, -np.inf

        for pos in remaining:
            redundancy = 0.0
            if selected:
                redundancy = float(np.max(
                    np.dot(candidate_embeddings[selected],
                           candidate_embeddings[pos])
                ))

            score = lambda_param * relevance[pos] - (1 - lambda_param) * redundancy

            if score > best_score:
                best_pos, best_score = pos, score

        selected.append(best_pos)
        remaining.remove(best_pos)

    rows = []
    for rank, pos in enumerate(selected, 1):
        idx = candidate_indices[pos]
        row = chunks_df.iloc[idx]
        rows.append({
            "mmr_rank": rank,
            "index": idx,
            "chunk_id": row["chunk_id"],
            "source_id": row["source_id"],
            "category": row["category"],
            "section": row["section"],
            "content": row["content"],
            "embedding_text": row["embedding_text"],
            "faiss_rank": candidates[pos]["faiss_rank"],
            "faiss_score": candidates[pos]["faiss_score"]
        })

    return pd.DataFrame(rows)


def mmr_then_rerank(question, candidate_k=10, mmr_k=5, lambda_param=0.70):
    candidates = mmr_select(
        question, candidate_k, mmr_k, lambda_param
    ).copy()

    pairs = [
        (question, row["embedding_text"])
        for _, row in candidates.iterrows()
    ]

    candidates["reranker_score"] = reranker.predict(pairs)

    candidates = candidates.sort_values(
        "reranker_score", ascending=False
    ).reset_index(drop=True)

    candidates["final_rank"] = np.arange(len(candidates)) + 1
    return candidates

Inspect the previously difficult questions


In [5]:
problem_questions = [
    (
        "Missed test",
        "I missed a test because I was unwell, How long do I have to apply for special consideration?",
        ["SC-01_8", "EAA-02_43"]
    ),
    (
        "Hard drive",
        "So my hard drive with my assignment stopped working and I didn't backup my work, will I be able to get an extension or special consideration?",
        ["SC-01_16"]
    ),
    (
        "ELP extension",
        "The extensions adjustment is on my ELP, but the assignment brief also says to fill out the standard extension form, do I need to do both, or is my learning plan enough on its own",
        ["EXT-01_2", "EAA-02_39"]
    )
]

for name, question, expected in problem_questions:
    print("\n" + "=" * 80)
    print(name, "| Expected:", expected)
    result = mmr_then_rerank(question)
    display(result[[
        "final_rank", "chunk_id", "mmr_rank",
        "faiss_rank", "faiss_score", "reranker_score"
    ]])


Missed test | Expected: ['SC-01_8', 'EAA-02_43']


,final_rank,chunk_id,mmr_rank,faiss_rank,faiss_score,reranker_score
0,1,EAA-02_43,5,6,0.478629,2.680190
1,2,SC-01_8,1,1,0.595475,2.300819
2,3,POL-01_63,3,2,0.571715,1.833305
3,4,EXT-01_0,4,3,0.493480,1.758704
4,5,SC-01_7,2,9,0.457805,-2.420846



Hard drive | Expected: ['SC-01_16']


,final_rank,chunk_id,mmr_rank,faiss_rank,faiss_score,reranker_score
0,1,SC-01_16,2,7,0.399607,-2.381408
1,2,SC-01_6,4,9,0.376300,-4.433515
2,3,EXT-01_0,5,2,0.471793,-4.844401
3,4,EXT-01_1,1,1,0.481436,-5.289862
4,5,EAA-02_38,3,5,0.415588,-6.256051



ELP extension | Expected: ['EXT-01_2', 'EAA-02_39']


,final_rank,chunk_id,mmr_rank,faiss_rank,faiss_score,reranker_score
0,1,EAA-02_41,3,6,0.568286,0.355364
1,2,EAA-02_37,4,3,0.587779,-0.159095
2,3,EAA-02_39,1,1,0.602267,-0.321993
3,4,EAA-02_40,5,5,0.575118,-1.383296
4,5,POL-01_62,2,8,0.555921,-4.155398


In [6]:
def any_expected(retrieved, expected):
    return any(x in retrieved for x in expected)

def all_expected(retrieved, expected):
    return all(x in retrieved for x in expected)

def evaluate_iteration8(test_df):
    results = []

    for _, test in test_df.iterrows():
        question = test["prompt"]
        expected = test["chunks"]

        retrieved_df = mmr_then_rerank(
            question, candidate_k=10, mmr_k=5, lambda_param=0.70
        )
        retrieved = retrieved_df["chunk_id"].tolist()

        top1, top3, top5 = retrieved[:1], retrieved[:3], retrieved[:5]
        ranks = [
            retrieved.index(x) + 1
            for x in expected if x in retrieved
        ]

        results.append({
            "prompt": question,
            "expected_chunks": expected,
            "top1_chunk": top1[0] if top1 else None,
            "top3_chunks": top3,
            "top5_chunks": top5,
            "expected_rank_in_top5": min(ranks) if ranks else None,
            "top1_hit": any_expected(top1, expected),
            "top3_hit": any_expected(top3, expected),
            "top5_hit": any_expected(top5, expected),
            "top1_all_hit": all_expected(top1, expected),
            "top3_all_hit": all_expected(top3, expected),
            "top5_all_hit": all_expected(top5, expected)
        })

    return pd.DataFrame(results)

iteration8_df = evaluate_iteration8(test_df)

print("ITERATION 8 — MMR + CROSS-ENCODER")
print("---------------------------------")
print("\nAny expected chunk retrieved:")
print(f"Top-1: {iteration8_df['top1_hit'].mean():.1%}")
print(f"Top-3: {iteration8_df['top3_hit'].mean():.1%}")
print(f"Top-5: {iteration8_df['top5_hit'].mean():.1%}")

print("\nAll expected chunks retrieved:")
print(f"Top-1: {iteration8_df['top1_all_hit'].mean():.1%}")
print(f"Top-3: {iteration8_df['top3_all_hit'].mean():.1%}")
print(f"Top-5: {iteration8_df['top5_all_hit'].mean():.1%}")

ITERATION 8 — MMR + CROSS-ENCODER
---------------------------------

Any expected chunk retrieved:
Top-1: 66.7%
Top-3: 83.3%
Top-5: 100.0%

All expected chunks retrieved:
Top-1: 58.3%
Top-3: 75.0%
Top-5: 83.3%


In [7]:
comparison_df = pd.DataFrame({
    "Metric": [
        "Any expected — Top-1", "Any expected — Top-3", "Any expected — Top-5",
        "All expected — Top-1", "All expected — Top-3", "All expected — Top-5"
    ],
    "Iteration 2": [0.500, 0.750, 0.917, 0.417, 0.583, 0.750],
    "Corrected Iteration 5": [0.583, 0.667, 0.917, 0.500, 0.667, 0.750],
    "Iteration 6": [0.500, 0.833, 0.917, 0.417, 0.667, 0.750],
    "Iteration 7": [0.500, 0.833, 1.000, 0.417, 0.500, 0.833],
    "Iteration 8": [
        iteration8_df["top1_hit"].mean(),
        iteration8_df["top3_hit"].mean(),
        iteration8_df["top5_hit"].mean(),
        iteration8_df["top1_all_hit"].mean(),
        iteration8_df["top3_all_hit"].mean(),
        iteration8_df["top5_all_hit"].mean()
    ]
})

comparison_df["I8 vs I7"] = comparison_df["Iteration 8"] - comparison_df["Iteration 7"]

display(comparison_df.style.format({
    "Iteration 2": "{:.1%}",
    "Corrected Iteration 5": "{:.1%}",
    "Iteration 6": "{:.1%}",
    "Iteration 7": "{:.1%}",
    "Iteration 8": "{:.1%}",
    "I8 vs I7": "{:+.1%}"
}))

,Metric,Iteration 2,Corrected Iteration 5,Iteration 6,Iteration 7,Iteration 8,I8 vs I7
0,Any expected — Top-1,50.0%,58.3%,50.0%,50.0%,66.7%,+16.7%
1,Any expected — Top-3,75.0%,66.7%,83.3%,83.3%,83.3%,+0.0%
2,Any expected — Top-5,91.7%,91.7%,91.7%,100.0%,100.0%,+0.0%
3,All expected — Top-1,41.7%,50.0%,41.7%,41.7%,58.3%,+16.6%
4,All expected — Top-3,58.3%,66.7%,66.7%,50.0%,75.0%,+25.0%
5,All expected — Top-5,75.0%,75.0%,75.0%,83.3%,83.3%,+0.0%


In [8]:
top5_failures_8 = iteration8_df[
    iteration8_df["top5_hit"] == False
][[
    "prompt", "expected_chunks", "top1_chunk",
    "top3_chunks", "top5_chunks", "expected_rank_in_top5"
]]

print("Iteration 8 Top-5 failures:", len(top5_failures_8))
display(top5_failures_8)

display(iteration8_df[[
    "prompt", "expected_chunks", "top1_chunk",
    "top3_chunks", "top5_chunks", "expected_rank_in_top5"
]])

Iteration 8 Top-5 failures: 0


,prompt,expected_chunks,top1_chunk,top3_chunks,top5_chunks,expected_rank_in_top5


,prompt,expected_chunks,top1_chunk,top3_chunks,top5_chunks,expected_rank_in_top5
0,I've got a timed quiz that I won't be able to ...,[EXT-01_1],EXT-01_1,"[EXT-01_1, EXT-01_0, EAA-02_38]","[EXT-01_1, EXT-01_0, EAA-02_38, EXT-01_2, POL-...",1
1,"I've recently lost my job, is this a valid rea...",[SC-01_7],SC-01_7,"[SC-01_7, SC-01_13, SC-01_4]","[SC-01_7, SC-01_13, SC-01_4, SC-01_8, POL-01_63]",1
2,So my hard drive with my assignment stopped wo...,[SC-01_16],SC-01_16,"[SC-01_16, SC-01_6, EXT-01_0]","[SC-01_16, SC-01_6, EXT-01_0, EXT-01_1, EAA-02...",1
3,I've just booked my first appointment with ELS...,[EAA-02_29],EAA-02_29,"[EAA-02_29, EAA-01_18, EAA-02_28]","[EAA-02_29, EAA-01_18, EAA-02_28, EAA-02_20, E...",1
4,I emailed my course coordinator three days ago...,[EAA-02_38],EAA-02_34,"[EAA-02_34, EAA-02_38, EXT-01_0]","[EAA-02_34, EAA-02_38, EXT-01_0, POL-01_62, SC...",2
5,Having a hard time reading through journal art...,[EAA-02_46],EAA-02_46,"[EAA-02_46, EAA-02_51, SC-01_15]","[EAA-02_46, EAA-02_51, SC-01_15, EAA-02_52, EA...",1
6,My mate got flagged for plagiarism because he ...,[POL-01_70],POL-01_70,"[POL-01_70, EXT-01_3, POL-01_56]","[POL-01_70, EXT-01_3, POL-01_56, EAA-02_19, PO...",1
7,Who actually sets RMIT's official academic cal...,[POL-01_67],POL-01_67,"[POL-01_67, EAA-02_50, SC-01_11]","[POL-01_67, EAA-02_50, SC-01_11, EAA-02_52, EX...",1
8,"I missed a test because I was unwell, How long...","[SC-01_8, EAA-02_43]",EAA-02_43,"[EAA-02_43, SC-01_8, POL-01_63]","[EAA-02_43, SC-01_8, POL-01_63, EAA-01_18, POL...",1
9,My nan just passed away but I've got a group a...,"[POL-01_62, EXT-01_1]",EAA-02_41,"[EAA-02_41, EAA-02_34, EXT-01_0]","[EAA-02_41, EAA-02_34, EXT-01_0, POL-01_62, EX...",4


In [9]:
iteration8_df.to_csv(
    "./11_iteration8_mmr_crossencoder_results.csv",
    index=False, encoding="utf-8-sig"
)
comparison_df.to_csv(
    "./11_iteration2_to_iteration8_comparison.csv",
    index=False, encoding="utf-8-sig"
)
print("Results saved.")

Results saved.
